# Stage 01 — Paper Intelligence

Read the paper PDF and raw data, then write `data/paper_spec.json`.
Follow `skills/stage_01.md` for detailed instructions.

In [1]:
from paths import *
import json, yaml
import pandas as pd

# Load config
config = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
print('Project root:', PROJECT_ROOT)
print('Config loaded:', list(config.keys()))

Project root: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs
Config loaded: ['paper', 'identification', 'dml', 'causal_forest', 'review']


In [2]:
# --- Inspect raw data variable names ---
# This paper uses CSV (not Stata), so load with pandas directly
primary_file = RAW_DATA_DIR / config['identification']['primary_data_file']
df = pd.read_csv(str(primary_file))
print(f'Shape: {df.shape}')
print(f'\nAll columns ({len(df.columns)}):')
for col in df.columns:
    print(f'  {col}')

# Verify all config variables exist in the data
controls = config['identification']['controls']
treatments = [config['identification']['treatment_variable']]
for t in config['identification'].get('additional_treatments', []):
    treatments.append(t['variable'])
outcome = config['identification']['outcome_variable']

all_vars = [outcome] + treatments + controls
print('\n=== Variable verification ===')
missing = [v for v in all_vars if v not in df.columns]
if missing:
    print(f'MISSING variables: {missing}')
else:
    print('All required variables present in data.')

# Verify country-level subsetting (industry == 211212)
# Note: in CSV the industry column is integer, not string
country_data = df[df['industry'] == 211212]
print(f'\nAfter filter industry==211212: {len(country_data)} rows')
complete = country_data.dropna(subset=all_vars)
print(f'After dropna on all vars: {len(complete)} rows (expected: 63)')

Shape: (1134, 53)

All columns (53):
  Unnamed: 0
  wbcode
  industry
  industry_description
  init_year
  final_year
  skill_2_1
  skill1_corr
  diffa
  diffg
  tskilla
  tnoskilla
  tskillg
  tnoskillg
  avg_tar
  init_tar
  prod_growth
  growth
  inv
  init
  human_cap
  dum80_83
  dum85_87
  lat_america
  e_europe
  w_africa
  e_africa
  s_c_africa
  n_afr_me
  e_asia
  s_e_asia
  s_w_asia
  ln_init_q
  ln_init_q_skilla
  ln_init_q_unskilla
  ln_init_q_skillg
  ln_init_q_unskillg
  voice2000
  political_stability2000
  govt_effectiveness2000
  reg_quality2000
  rule_of_law2000
  control_corrup2000
  growth_human_cap
  patent_growth
  diff_growth_q_a
  diff_growth_q_g
  fin_diffa
  fin_diffg
  percent_firms_connected
  ln_viol_per_car
  gatt_year
  wto_year

=== Variable verification ===
All required variables present in data.

After filter industry==211212: 63 rows
After dropna on all vars: 63 rows (expected: 63)


In [3]:
# --- Build paper_spec from R code ground truth and paper context ---
#
# Source of truth: dml_tariffs_T2.R (lines 51-56), data_tariffs_table_column.R,
# GRF_tariffs_Table4.Rmd — all from B&N replication package.
#
# Key facts from R code:
#   data <- data.raw[which(data.raw$industry=="211212"),]  -> 63 country-level obs
#   y <- "growth"
#   d <- "skill1_corr" (Panel A), "diffa" (Panel B), "diffg" (Panel C)
#   17 controls: avg_tar, init, inv, human_cap, w_africa, e_africa, s_c_africa,
#                n_afr_me, e_europe, lat_america, e_asia, s_e_asia, s_w_asia,
#                dum80_83, dum85_87, ln_init_q_skilla, ln_init_q_unskilla
#   est="plinear" (Partial Linear Regression), z=NULL (no instrument -> PLR not PLIV)
#   DML="DML2", nfold=2, ite=100

# Reference results from B&N Table 2 (country-level):
#   Panel A (skill1_corr): OLS=0.035, DML Lasso=0.019, DML Boosting=0.016, DML Forest=0.016
#   Panel B (diffa):       OLS=0.016, DML Lasso=0.010, DML Forest=0.008
#   Panel C (diffg):       OLS=0.020, DML Lasso=0.009, DML Forest=0.008

paper_spec = {
    "title": "The Structure of Tariffs and Long-term Growth",
    "authors": ["Nunn, N.", "Trefler, D."],
    "year": 2010,
    "journal": "American Economic Journal: Macroeconomics",
    "slug": "nunntrefler2010",

    "identification": {
        "type": "OLS",
        "narrative": (
            "Nunn and Trefler (2010) examine whether skill-biased tariff protection "
            "affects long-term economic growth. Using cross-country data, they regress "
            "log annual per capita GDP growth on measures of tariff skill-bias "
            "(skill1_corr, diffa, diffg) controlling for initial GDP, investment, human "
            "capital, average tariff level, and region dummies. The country-level sample "
            "(industry=='211212') contains 63 observations. There is no instrument; the "
            "identification relies on selection-on-observables with a rich set of 17 "
            "controls including regional dummies and initial skill-intensity measures. "
            "Baiardi and Naghi (2024) revisit this paper using Double/Debiased ML "
            "(DML2 with 2-fold cross-fitting, 100 repetitions) and find that DML "
            "estimates are substantially smaller than OLS, suggesting OLS suffers from "
            "regularization bias when the number of controls is large relative to n=63."
        ),
        "outcome_variable": "growth",
        "outcome_label": "Log annual per capita GDP growth",
        "treatment_variable": "skill1_corr",
        "treatment_label": "Skill-tariff correlation (Table 2 Panel A primary treatment)",
        "instrument": None,
        "instrument_label": None,
        "functional_form": "linear",
        "controls": [
            "avg_tar",
            "init",
            "inv",
            "human_cap",
            "w_africa",
            "e_africa",
            "s_c_africa",
            "n_afr_me",
            "e_europe",
            "lat_america",
            "e_asia",
            "s_e_asia",
            "s_w_asia",
            "dum80_83",
            "dum85_87",
            "ln_init_q_skilla",
            "ln_init_q_unskilla"
        ],
        "fixed_effects": [],
        "cluster_se": None,
        "primary_data_file": "data_tariffs.csv",
        "secondary_datasets": [],
        "data_filter": "industry == 211212",
        "additional_treatments": [
            {
                "variable": "diffa",
                "label": "Tariff differential (low skill cut-off, Table 2 Panel B)"
            },
            {
                "variable": "diffg",
                "label": "Tariff differential (high skill cut-off, Table 2 Panel C)"
            }
        ]
    },

    "main_results": [
        {
            "table": "Table 2, Panel A",
            "spec": "OLS — growth on skill1_corr, 17 controls, country-level",
            "coef": 0.035,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "OLS baseline from Nunn & Trefler (2010) Table 2 Panel A"
        },
        {
            "table": "B&N Table 2, Panel A — DML Lasso",
            "spec": "DML2 PLR — growth on skill1_corr, Lasso, 2-fold, 100 reps",
            "coef": 0.019,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        },
        {
            "table": "B&N Table 2, Panel A — DML Boosting",
            "spec": "DML2 PLR — growth on skill1_corr, Boosting, 2-fold, 100 reps",
            "coef": 0.016,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        },
        {
            "table": "B&N Table 2, Panel A — DML Forest",
            "spec": "DML2 PLR — growth on skill1_corr, Forest, 2-fold, 100 reps",
            "coef": 0.016,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        },
        {
            "table": "B&N Table 2, Panel B — OLS",
            "spec": "OLS — growth on diffa, 17 controls, country-level",
            "coef": 0.016,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "OLS baseline, Panel B treatment=diffa"
        },
        {
            "table": "B&N Table 2, Panel B — DML Lasso",
            "spec": "DML2 PLR — growth on diffa, Lasso, 2-fold, 100 reps",
            "coef": 0.010,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        },
        {
            "table": "B&N Table 2, Panel B — DML Forest",
            "spec": "DML2 PLR — growth on diffa, Forest, 2-fold, 100 reps",
            "coef": 0.008,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        },
        {
            "table": "B&N Table 2, Panel C — OLS",
            "spec": "OLS — growth on diffg, 17 controls, country-level",
            "coef": 0.020,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "OLS baseline, Panel C treatment=diffg"
        },
        {
            "table": "B&N Table 2, Panel C — DML Lasso",
            "spec": "DML2 PLR — growth on diffg, Lasso, 2-fold, 100 reps",
            "coef": 0.009,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        },
        {
            "table": "B&N Table 2, Panel C — DML Forest",
            "spec": "DML2 PLR — growth on diffg, Forest, 2-fold, 100 reps",
            "coef": 0.008,
            "se": None,
            "ci_lo": None,
            "ci_hi": None,
            "n_obs": 63,
            "notes": "Baiardi & Naghi (2024) DML replication; median over 100 repetitions"
        }
    ],

    "dml": {
        "model_type": "PLR",
        "rationale": (
            "Nunn & Trefler (2010) use OLS with no instrument (z=NULL in the R code), "
            "making Partial Linear Regression (PLR / 'plinear') the correct DoubleML model. "
            "B&N confirm this by setting est='plinear' in their DML code. The treatment is "
            "continuous and the identification is selection-on-observables."
        )
    }
}

print('=== paper_spec built ===')
print(f'Title: {paper_spec["title"]}')
print(f'Main results: {len(paper_spec["main_results"])} entries')
print(f'Controls: {len(paper_spec["identification"]["controls"])} variables')
print(f'DML model: {paper_spec["dml"]["model_type"]}')
print(f'Treatments: {paper_spec["identification"]["treatment_variable"]}, '
      + ', '.join(t["variable"] for t in paper_spec["identification"]["additional_treatments"]))

=== paper_spec built ===
Title: The Structure of Tariffs and Long-term Growth
Main results: 10 entries
Controls: 17 variables
DML model: PLR
Treatments: skill1_corr, diffa, diffg


In [4]:
# --- Write paper_spec.json (atomic write) ---
import shutil

tmp = PAPER_SPEC.with_suffix('.json.tmp')
tmp.write_text(json.dumps(paper_spec, indent=2), encoding='utf-8')
shutil.move(str(tmp), str(PAPER_SPEC))
print(f'Written: {PAPER_SPEC}')

# Verify by round-tripping
loaded = json.loads(PAPER_SPEC.read_text(encoding='utf-8'))
assert loaded['title'] == paper_spec['title'], 'Title mismatch after round-trip!'
assert len(loaded['main_results']) >= 3, f'Expected >=3 main results, got {len(loaded["main_results"])}'
assert loaded['identification']['outcome_variable'] == 'growth'
assert loaded['identification']['treatment_variable'] == 'skill1_corr'
assert loaded['identification']['instrument'] is None
assert len(loaded['identification']['controls']) == 17, f'Expected 17 controls, got {len(loaded["identification"]["controls"])}'
print('Verification passed.')
print()
print(json.dumps(loaded, indent=2)[:2000])

Written: C:\Users\qgallea\Dropbox\work\claude_code\recast\papers\05_nunn_trefler_tariffs\data\paper_spec.json
Verification passed.

{
  "title": "The Structure of Tariffs and Long-term Growth",
  "authors": [
    "Nunn, N.",
    "Trefler, D."
  ],
  "year": 2010,
  "journal": "American Economic Journal: Macroeconomics",
  "slug": "nunntrefler2010",
  "identification": {
    "type": "OLS",
    "narrative": "Nunn and Trefler (2010) examine whether skill-biased tariff protection affects long-term economic growth. Using cross-country data, they regress log annual per capita GDP growth on measures of tariff skill-bias (skill1_corr, diffa, diffg) controlling for initial GDP, investment, human capital, average tariff level, and region dummies. The country-level sample (industry=='211212') contains 63 observations. There is no instrument; the identification relies on selection-on-observables with a rich set of 17 controls including regional dummies and initial skill-intensity measures. Baiardi